#**CS5100 - Foundations of AI**
##**Fall 2025 Homework Assignment 4**

Submission Instructions:
- Please complete this homework assignment in the same notebook provided.
- Submit your completed assignment on Canvas by the deadline.

Submission Deadline:
**October 20th, 2025**

<p align="justify">
Please read the instructions carefully when answering questions and ensure your code works correctly before submission. The grader will run your code for grading the coding questions without any adjustment.
</p>

In [ ]:
#@markdown ### Enter your first and last names below:
First_Name = "Xinle" #@param {type:"string"}
Last_Name = "Xu" #@param {type:"string"}

#**Analytical and Implementation**
#**(50 points) Question 1: Probabilistic Framework for Logistic Regression**

In lecture 7, we explored the probabilistic framework of logistic regression, encompassing both Maximum Likelihood Estimation (MLE) and Maximum a Posteriori (MAP) estimation. In this assignment, you are required to derive the appropriate mathematical formulations and subsequently implement them for both binary and multi-class logistic regression.

**1. Binary Classification Using Logistic Regression:**

Given the probabilistic representation:
$$p(y = 1 | \mathbf{x}; \mathbf{w}) = \sigma(\mathbf{w}^T \mathbf{x}) $$
Where:
- $ \mathbf{w} $ is the weight vector.
- $ \mathbf{x} $ is the input feature vector.
- $ \sigma(z) $ is the sigmoid function given by $ \sigma(z) = \frac{1}{1 + e^{-z}} $.

a. **MLE for Binary Classification (10 points)**:
   - Derive the formula for the log-likelihood for binary logistic regression.
   - Implement a Python function based on your derived formula.
   - Using the provided code, compute and report the MLE solution for the binary classification problem.

b. **MAP for Binary Classification (10 points)**:
   - Derive the formula for the log-posterior, considering a Gaussian prior on the weights.
   - Implement a Python function based on your derived formula.
   - Using the provided code, compute and report the MAP solution for the binary classification problem.

**2. Multi-Class Classification Using Softmax Regression:**

Given the probability representation for class $ c $ using the softmax function:
$$ p(y=c | \mathbf{x}; \mathbf{W}) = \frac{e^{\mathbf{w}_c^T \mathbf{x}}} {\sum_{j=1}^{C} e^{\mathbf{w}_j^T \mathbf{x}}} $$
Where:
- $ \mathbf{W} $ is a matrix where each column is the weight vector for a particular class.
- $ \mathbf{w}_c $ is the weight vector for class $ c $.

a. **MLE for Multi-Class Classification (10 points)**:
   - Derive the formula for the log-likelihood for multi-class logistic regression using the softmax function.
   - Implement a Python function based on your derived formula.
   - Using the provided code, compute and report the MLE solution for the multi-class classification problem.

b. **MAP for Multi-Class Classification (10 points)**:
   - Derive the formula for the log-posterior for multi-class logistic regression, considering a Gaussian prior on the weights for each class.
   - Implement a Python function based on your derived formula.
   - Using the provided code, compute and report the MAP solution for the multi-class classification problem.

**3. Analysis (10 points)**:
   - Compare and contrast the results obtained from MLE and MAP solutions for both binary and multi-class logistic regression. Discuss the influence of the Gaussian prior in the MAP solutions.


**Starter Code**:




1(a) The binary log-likelihood is $\ell(\mathbf w)=\sum_i\big[y_i\,\mathbf w^\top\mathbf x_i-\log(1+e^{\mathbf w^\top\mathbf x_i})\big]$. This is the quantity maximized by MLE.

1(b) With a zero-mean Gaussian prior $\mathcal N(\mu,\sigma^2\mathbf I)$ on $\mathbf w$, the log-posterior is $\log p(\mathbf w\mid\mathcal D)=\ell(\mathbf w)-\tfrac{1}{2\sigma^2}\lVert\mathbf w-\mu\rVert_2^2 + C$. On the provided demo data, the MLE solution is $w_{\text{MLE}}\approx [1.682114, 7.446978]$ with log-likelihood -0.007612; the MAP solution (mu=0, sigma=1) is $w_{\text{MAP}}\approx [0.096707, 0.404213]$ with log-posterior -1.532071.

2(a) For multi-class softmax with class weights $\{\mathbf w_k\}_{k=1}^K$ and $p(y=k\mid\mathbf x)=\dfrac{\exp(\mathbf w_k^\top\mathbf x)}{\sum_j\exp(\mathbf w_j^\top\mathbf x)}$, the log-likelihood is $\ell(\mathbf W)=\sum_i\Big(\mathbf w_{y_i}^\top\mathbf x_i-\log\sum_{k=1}^K e^{\mathbf w_k^\top\mathbf x_i}\Big)$. On the demo data, the MLE solution is $W_{\text{MLE}}\approx$ 

```
[
  [1.618223, -0.310632, -1.307591],
  [-1.332804, -3.918139, 5.250943],
  [-6.181757, 3.270289, 2.911468]
]
```
with log-likelihood -0.005920.

2(b) Adding the same Gaussian prior per weight gives the MAP objective $\log p(\mathbf W\mid\mathcal D)=\ell(\mathbf W)-\tfrac{1}{2\sigma^2}\sum_k\lVert\mathbf w_k-\mu\rVert_2^2 + C$. On the demo data (mu=0, sigma=1), the MAP solution is $W_{\text{MAP}}\approx$ 

```
[
  [0.012174, -0.000721, -0.011454],
  [0.002491, -0.240224, 0.237734],
  [-0.209890, 0.170310, 0.039580]
]
```
with log-posterior -2.776465.

3. Across both settings, MLE focuses only on fitting the data and can drive weights large and probabilities extreme when data are small or noisy; MAP adds an $L2$-style shrinkage from the Gaussian prior, pulling weights toward $0$, which usually improves calibration and generalization. Smaller $\sigma^2$ means stronger shrinkage (further from MLE); large $\sigma^2$ makes MAP approach MLE.

In [ ]:
import numpy as np

# Sigmoid function for binary classification
def sigmoid(z):
    """Numerically stable sigmoid that supports scalars, vectors, and matrices."""
    z = np.asarray(z)
    # Use a stable formulation: sigmoid(z) = 1 / (1 + exp(-z)), piecewise to avoid overflow
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    neg = ~pos
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    exp_z = np.exp(z[neg])  # safe because z[neg] < 0
    out[neg] = exp_z / (1.0 + exp_z)
    return out

# Softmax function for multi-class classification
def softmax(z):
    """Numerically stable softmax.
    If z is 1D, returns a 1D probability vector.
    If z is 2D (n,d) returns (n,d) with row-wise softmax.
    """
    Z = np.asarray(z, dtype=float)
    if Z.ndim == 1:
        m = Z.max()
        exp = np.exp(Z - m)
        return exp / exp.sum()
    elif Z.ndim == 2:
        m = Z.max(axis=1, keepdims=True)
        exp = np.exp(Z - m)
        return exp / exp.sum(axis=1, keepdims=True)
    else:
        raise ValueError("softmax expects 1D or 2D input")

# Binary log-likelihood
def binary_log_likelihood(w, X, y):
    """Log-likelihood for binary logistic regression.
    w: (d,)
    X: (n,d)
    y: (n,) in {0,1}
    Returns a scalar log-likelihood.
    """
    w = np.asarray(w, dtype=float)
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    z = X @ w
    # Stable: y*log(sigmoid(z)) + (1-y)*log(1-sigmoid(z))
    ll = - (y * np.logaddexp(0, -z) + (1 - y) * np.logaddexp(0, z)).sum()
    return ll

# Multi-class log-likelihood
def multi_log_likelihood(W, X, y_onehot):
    """Log-likelihood for multi-class softmax regression.
    W: (d, K)
    X: (n, d)
    y_onehot: (n, K) one-hot rows
    Returns a scalar log-likelihood.
    """
    W = np.asarray(W, dtype=float)
    X = np.asarray(X, dtype=float)
    Y = np.asarray(y_onehot, dtype=float)
    logits = X @ W  # (n,K)
    # log-sum-exp for stability
    lse = np.log(np.exp(logits - logits.max(axis=1, keepdims=True)).sum(axis=1, keepdims=True)) + logits.max(axis=1, keepdims=True)
    # Sum_i sum_k y_ik * (log p_ik) = sum_i sum_k y_ik * (logits_ik - lse_i)
    ll = (Y * (logits - lse)).sum()
    return ll
...
# Log-prior for an isotropic Gaussian N(mu, sigma^2 I)
def gaussian_log_prior(theta, mu, sigma):
    theta = np.asarray(theta, dtype=float)
    # Ignore constants (-(d/2)log(2*pi*sigma^2)) since comparisons are relative
    return -0.5 * np.sum(((theta - mu) / sigma) ** 2)

# Binary log-posterior (MAP objective up to an additive constant)
def binary_log_posterior(w, X, y, mu, sigma):
    return binary_log_likelihood(w, X, y) + gaussian_log_prior(w, mu, sigma)

# Multi-class log-posterior (sum of class-wise priors)
def multi_log_posterior(W, X, y_onehot, mu, sigma):
    return multi_log_likelihood(W, X, y_onehot) + gaussian_log_prior(W, mu, sigma)

# --- Minimal test data so the provided tests below produce numeric outputs ---
# Binary: include a bias column in X
X_bin = np.array([
    [1.0,  0.5],
    [1.0, -1.0],
    [1.0,  2.0],
])
y_bin = np.array([1, 0, 1])
w_bin = np.array([0.3, -0.7])

# Multi-class: 3 classes, include a bias column
X_multi = np.array([
    [1.0,  0.2, -0.5],
    [1.0,  1.0,  0.3],
    [1.0, -0.7,  0.8],
])
y_multi = np.array([0, 2, 1])  # class indices
K = 3
y_multi_onehot = np.eye(K)[y_multi]
W_multi = np.array([
    [ 0.1, -0.2,  0.0],
    [ 0.5,  0.3, -0.4],
    [-0.3,  0.2,  0.6],
])

mu = 0
sigma = 1

# Test Binary Log-Posterior Function (for MAP)
print("=== Testing Binary Log-Posterior Function (for MAP) ===")
print("Input Weights:", w_bin)
print("Result:", binary_log_posterior(w_bin, X_bin, y_bin, mu, sigma))
print("\n")

# Test Multi-class Log-Posterior Function (for MAP)
print("=== Testing Multi-class Log-Posterior Function (for MAP) ===")
print("Input Weights:\n", W_multi)
print("Result:", multi_log_posterior(W_multi, X_multi, y_multi_onehot, mu, sigma))



# -----------------------------
# Simple optimizers (MLE and MAP) for reporting solutions
# -----------------------------

def fit_logreg_binary_mle(X, y, lr=0.2, iters=2000):
    """Return w that maximizes the binary log-likelihood (no regularization)."""
    n, d = X.shape
    w = np.zeros(d)
    for t in range(iters):
        p = sigmoid(X @ w)
        grad = X.T @ (y - p) / n
        w += lr * grad
    return w

def fit_logreg_binary_map(X, y, mu, sigma, lr=0.2, iters=2000):
    """Return w that maximizes log-posterior with N(mu, sigma^2 I) prior."""
    n, d = X.shape
    w = np.zeros(d)
    mu_vec = np.zeros(d) + mu  # allow scalar mu
    lam = 1.0 / (sigma ** 2)
    for t in range(iters):
        p = sigmoid(X @ w)
        grad = X.T @ (y - p) / n - lam * (w - mu_vec)
        w += lr * grad
    return w

def fit_softmax_mle(X, y_onehot, lr=0.2, iters=3000):
    """Return W that maximizes the multinomial log-likelihood (no regularization)."""
    n, d = X.shape
    K = y_onehot.shape[1]
    W = np.zeros((d, K))
    for t in range(iters):
        logits = X @ W
        P = softmax(logits)  # (n,K)
        grad = X.T @ (y_onehot - P) / n
        W += lr * grad
    return W

def fit_softmax_map(X, y_onehot, mu, sigma, lr=0.2, iters=3000):
    """Return W that maximizes log-posterior with N(mu, sigma^2 I) prior (per weight)."""
    n, d = X.shape
    K = y_onehot.shape[1]
    W = np.zeros((d, K))
    mu_mat = np.zeros((d, K)) + mu
    lam = 1.0 / (sigma ** 2)
    for t in range(iters):
        logits = X @ W
        P = softmax(logits)  # (n,K)
        grad = X.T @ (y_onehot - P) / n - lam * (W - mu_mat)
        W += lr * grad
    return W

# -----------------------------
# Report MLE / MAP solutions on the tiny demo data above
# -----------------------------
print("\n=== Reported Binary MLE solution ===")
w_mle_bin = fit_logreg_binary_mle(X_bin, y_bin, lr=0.3, iters=2500)
print("w_MLE:", np.round(w_mle_bin, 4))
print("log-likelihood:", binary_log_likelihood(w_mle_bin, X_bin, y_bin))

print("\n=== Reported Binary MAP solution (mu=0, sigma=1) ===")
w_map_bin = fit_logreg_binary_map(X_bin, y_bin, mu=mu, sigma=sigma, lr=0.3, iters=2500)
print("w_MAP:", np.round(w_map_bin, 4))
print("log-posterior:", binary_log_posterior(w_map_bin, X_bin, y_bin, mu, sigma))

print("\n=== Reported Multi-class MLE solution ===")
W_mle_multi = fit_softmax_mle(X_multi, y_multi_onehot, lr=0.3, iters=4000)
print("W_MLE:\n", np.round(W_mle_multi, 4))
print("log-likelihood:", multi_log_likelihood(W_mle_multi, X_multi, y_multi_onehot))

print("\n=== Reported Multi-class MAP solution (mu=0, sigma=1) ===")
W_map_multi = fit_softmax_map(X_multi, y_multi_onehot, mu=mu, sigma=sigma, lr=0.3, iters=4000)
print("W_MAP:\n", np.round(W_map_multi, 4))
print("log-posterior:", multi_log_posterior(W_map_multi, X_multi, y_multi_onehot, mu, sigma))
